# $\alpha,\!\beta$-CROWN for Control Applications

### **Tutorial page: [abcrown.org/tutorial](https://abcrown.org/tutorial)**

Control examples covered here:

- Reachable-set bounding for nonlinear dynamics.
- Lyapunov stability and forward-invariance verification.
- Bounded nonlinear control optimization.
- Nonlinear satisfiability checking through the dReal-compatible interface.

## Setup
### Colab GPU Environment

Select **Runtime > Change runtime type > GPU** before executing the
installation cell.

<p align="center">
  <img
    src="https://github.com/Verified-Intelligence/abCROWN_Control_Tutorial/blob/main/figures/colab_setup.png?raw=1"
    alt="Colab GPU runtime setup"
    width="48%"
  />
</p>

In [ ]:
%%capture
# Work from $HOME and remove any previous clone.
%cd $HOME
!rm -rf alpha-beta-CROWN

# Clone submodules together with the main repository.
!git clone --recursive https://github.com/Verified-Intelligence/alpha-beta-CROWN.git
%cd alpha-beta-CROWN
!uv pip install --system -e .

## Verifier Workflow Review

The verifier workflow has four steps.

1. **Implement the nonlinear function as an `nn.Module`.** Write the map to analyze as an
   `nn.Module`; `forward` uses `torch` operations and returns the tensor output.
2. **Create symbolic variables, solver settings, and the solver.**
   `input_vars(...)` and `output_vars(...)` define symbolic input and output
   vectors. `ConfigBuilder` stores solver settings. `ABCrownSolver` binds the
   model, symbolic variables, and solver settings together.
3. **Specify constraints with Python expressions.** `IOConstraints` stores
   input domains and output properties written with indexing, arithmetic,
   comparisons, parentheses, `&`, and `|`.
4. **Call the solver.** The solver provides calls such as `compute_bounds(...)`,
   `verify(...)`, `minimize(...)`, and `maximize(...)` for the requested task.


In [ ]:
#@title Runtime output helpers { display-mode: "form" }
!uv pip install --system matplotlib > /dev/null

import contextlib
import functools
import logging
import os
import warnings

@contextlib.contextmanager
def _quiet_abcrown_calls():
    with open(os.devnull, "w") as devnull:
        with contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
            yield

with _quiet_abcrown_calls(), warnings.catch_warnings():
    warnings.simplefilter("ignore", DeprecationWarning)
    import abcrown as _abcrown

logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("PIL").setLevel(logging.WARNING)

if not hasattr(_abcrown.ABCrownSolver, "_quiet_original_compute_bounds"):
    _abcrown.ABCrownSolver._quiet_original_compute_bounds = _abcrown.ABCrownSolver.compute_bounds
if not hasattr(_abcrown.ABCrownSolver, "_quiet_original_verify"):
    _abcrown.ABCrownSolver._quiet_original_verify = _abcrown.ABCrownSolver.verify
if hasattr(_abcrown.ABCrownSolver, "minimize") and not hasattr(_abcrown.ABCrownSolver, "_quiet_original_minimize"):
    _abcrown.ABCrownSolver._quiet_original_minimize = _abcrown.ABCrownSolver.minimize
if hasattr(_abcrown.ABCrownSolver, "maximize") and not hasattr(_abcrown.ABCrownSolver, "_quiet_original_maximize"):
    _abcrown.ABCrownSolver._quiet_original_maximize = _abcrown.ABCrownSolver.maximize
if not hasattr(_abcrown.IOConstraints, "_quiet_original_init"):
    _abcrown.IOConstraints._quiet_original_init = _abcrown.IOConstraints.__init__

@functools.wraps(_abcrown.ABCrownSolver._quiet_original_compute_bounds)
def _quiet_compute_bounds(self, *args, **kwargs):
    with _quiet_abcrown_calls():
        return _abcrown.ABCrownSolver._quiet_original_compute_bounds(self, *args, **kwargs)

@functools.wraps(_abcrown.ABCrownSolver._quiet_original_verify)
def _quiet_verify(self, *args, **kwargs):
    with _quiet_abcrown_calls():
        return _abcrown.ABCrownSolver._quiet_original_verify(self, *args, **kwargs)

if hasattr(_abcrown.ABCrownSolver, "_quiet_original_minimize"):
    @functools.wraps(_abcrown.ABCrownSolver._quiet_original_minimize)
    def _quiet_minimize(self, *args, **kwargs):
        with _quiet_abcrown_calls():
            return _abcrown.ABCrownSolver._quiet_original_minimize(self, *args, **kwargs)

if hasattr(_abcrown.ABCrownSolver, "_quiet_original_maximize"):
    @functools.wraps(_abcrown.ABCrownSolver._quiet_original_maximize)
    def _quiet_maximize(self, *args, **kwargs):
        with _quiet_abcrown_calls():
            return _abcrown.ABCrownSolver._quiet_original_maximize(self, *args, **kwargs)

@functools.wraps(_abcrown.IOConstraints._quiet_original_init)
def _quiet_ioconstraints_init(self, *args, **kwargs):
    with _quiet_abcrown_calls():
        return _abcrown.IOConstraints._quiet_original_init(self, *args, **kwargs)

_abcrown.ABCrownSolver.compute_bounds = _quiet_compute_bounds
_abcrown.ABCrownSolver.verify = _quiet_verify
if "_quiet_minimize" in globals():
    _abcrown.ABCrownSolver.minimize = _quiet_minimize
if "_quiet_maximize" in globals():
    _abcrown.ABCrownSolver.maximize = _quiet_maximize
_abcrown.IOConstraints.__init__ = _quiet_ioconstraints_init

In [ ]:
import torch
import torch.nn as nn

from abcrown import (
    ABCrownSolver,
    ConfigBuilder,
    IOConstraints,
    input_vars,
    output_vars,
)

torch.manual_seed(0)
torch.set_default_dtype(torch.float32)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required.")

DEVICE = "cuda"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
#@title Helper functions { display-mode: "form" }
# File, numerical, plotting, and checkpoint utilities used by examples.
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 15,
    "figure.titlesize": 20,
})
from matplotlib.patches import Rectangle
from matplotlib.ticker import MaxNLocator

# Auxiliary plotting code.
def as_numpy(value):
    if isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)

def sample_rectangle(lower, upper, steps=121):
    lower = torch.as_tensor(lower, dtype=torch.get_default_dtype(), device=DEVICE)
    upper = torch.as_tensor(upper, dtype=torch.get_default_dtype(), device=DEVICE)
    axes = [
        torch.linspace(lower[i], upper[i], steps=steps, device=DEVICE)
        for i in range(lower.numel())
    ]
    mesh = torch.meshgrid(*axes, indexing="ij")
    return torch.stack([axis.reshape(-1) for axis in mesh], dim=1)

def plot_reachability(model, lower, upper, outer_lower, outer_upper):
    points = sample_rectangle(lower, upper, steps=121)
    with torch.no_grad():
        outputs = model(points)

    outputs = as_numpy(outputs)
    outer_lower = as_numpy(outer_lower)
    outer_upper = as_numpy(outer_upper)

    fig, ax = plt.subplots(figsize=(8.5, 5.2))
    ax.scatter(
        outputs[:, 0],
        outputs[:, 1],
        s=5,
        alpha=0.35,
        label=r"Samples from $f(\mathcal{X})$",
    )
    ax.add_patch(
        Rectangle(
            outer_lower,
            outer_upper[0] - outer_lower[0],
            outer_upper[1] - outer_lower[1],
            fill=False,
            edgecolor="tab:red",
            linewidth=2.5,
            label="Sound outer bound",
        )
    )
    ax.set(
        xlabel="$q^+$",
        ylabel="$v^+$",
        title=r"One-step reachable set",
    )
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

def plot_optimization_bounds(primal_values, dual_bounds):
    primal_values = as_numpy(primal_values)
    dual_bounds = as_numpy(dual_bounds)
    iterations = np.arange(1, len(primal_values) + 1)

    fig, ax = plt.subplots(figsize=(8.5, 5.0))
    ax.plot(
        iterations,
        primal_values,
        marker="o",
        linewidth=2,
        label="Primal value (feasible upper bound)",
    )
    ax.plot(
        iterations,
        dual_bounds,
        marker="s",
        linewidth=2,
        label="Dual bound (certified lower bound)",
    )
    ax.set(
        xlabel="Input BaB iteration",
        ylabel="objective value",
        title="Primal and dual bound convergence",
    )
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(alpha=0.25)
    ax.legend()
    plt.tight_layout()
    plt.show()

def load_pendulum_lyapunov_model(model):
    checkpoint_path = (
        Path("neural_lyapunov_dependency")
        / "pendulum_discrete_state_feedback_quadratic.pt"
    )
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    if not checkpoint_path.exists():
        checkpoint_url = (
            "https://raw.githubusercontent.com/"
            "Verified-Intelligence/abCROWN_Control_Tutorial/main/"
            "neural_lyapunov_dependency/"
            "pendulum_discrete_state_feedback_quadratic.pt"
        )
        urllib.request.urlretrieve(checkpoint_url, checkpoint_path)

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["state_dict"], strict=False)
    return model.eval()

class ContinuousVanDerPolDynamics:
    """Controlled continuous-time Van der Pol dynamics."""

    def __init__(self, mu=1.0):
        self.mu = mu
        self.x_equilibrium = torch.zeros(2)
        self.u_equilibrium = torch.zeros(1)

    def f_torch(self, x, u):
        x1 = x[:, 0:1]
        x2 = x[:, 1:2]
        dx1 = x2
        dx2 = -x1 + self.mu * (1.0 - x1.square()) * x2 + u
        return torch.cat((dx1, dx2), dim=1)

class ContinuousTanhController(nn.Module):
    """Tanh controller architecture used by the trained continuous example."""

    def __init__(self, dims, x_equilibrium, u_equilibrium, scale=1.0):
        super().__init__()
        self.scale = scale
        layers = []
        for i in range(len(dims) - 2):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(dims[-2], dims[-1]))
        self.layers = nn.Sequential(*layers)
        self.register_buffer("x_equilibrium", x_equilibrium)
        self.register_buffer("u_equilibrium", u_equilibrium)
        self.register_buffer("b", torch.atanh(self.u_equilibrium / self.scale))

    def forward(self, x):
        x_eq = self.x_equilibrium.unsqueeze(0)
        u_eq = self.layers(x_eq)[0]
        u_diff = self.layers(x) - u_eq
        return self.scale * torch.tanh(u_diff + self.b)

class ContinuousNeuralLyapunov(nn.Module):
    """Neural Lyapunov value network used by the trained continuous example."""

    def __init__(self, dims):
        super().__init__()
        layers = []
        for i in range(len(dims) - 2):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            layers.append(nn.Tanh())
        layers.append(nn.Linear(dims[-2], dims[-1]))
        layers.append(nn.Sigmoid())
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)

def load_continuous_vanderpol_lyapunov_model(model):
    checkpoint_path = (
        Path("neural_lyapunov_dependency")
        / "van_der_pol_continuous_lyapunov.pt"
    )
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

    if not checkpoint_path.exists():
        checkpoint_url = (
            "https://raw.githubusercontent.com/"
            "Verified-Intelligence/alpha-beta-CROWN/main/"
            "complete_verifier/examples_abcrown/neural_lyapunov_dependency/"
            "seed_0.pth"
        )
        urllib.request.urlretrieve(checkpoint_url, checkpoint_path)

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    state_dict = checkpoint["state_dict"] if isinstance(checkpoint, dict) and "state_dict" in checkpoint else checkpoint
    model.load_state_dict(state_dict, strict=False)
    return model.eval()

def build_continuous_vanderpol_lyapunov_model(condition_cls):
    dynamics = ContinuousVanDerPolDynamics()
    controller = ContinuousTanhController(
        dims=[2, 10, 10, 1],
        x_equilibrium=dynamics.x_equilibrium,
        u_equilibrium=dynamics.u_equilibrium,
        scale=1.0,
    )
    lyapunov = ContinuousNeuralLyapunov(dims=[2, 40, 40, 1])
    condition = condition_cls(dynamics, controller, lyapunov).to(DEVICE)
    return load_continuous_vanderpol_lyapunov_model(condition)


## Application 1: Reachability

Reachability is an application of $\alpha,\!\beta$-CROWN bound computation.
The transition map is represented as a nonlinear function, the input set is encoded
as constraints over symbolic state variables, and `compute_bounds(...)` returns
a sound coordinatewise outer bound for $f(\mathcal{X})$.

For a discrete-time nonlinear transition map $x^+=f(x)$, reachability bounds enclose all next
states generated by an input set:

$$
f(\mathcal{X})=\{f(x):x\in\mathcal{X}\}.
$$

<p align="center"><img src="https://github.com/Verified-Intelligence/abCROWN_Control_Tutorial/blob/main/figures/tutorial_reachability_problem.png?raw=1" alt="Reachability illustration" width="50%" /></p>

For $\mu=1$, one Van der Pol Euler step with step size $0.1$ is

$$
q^+=q+0.1v,\qquad
v^+=v+0.1\big((1-q^2)v-q\big).
$$

The input set is

$$
\mathcal{X}=[-3,3]\times[-0.55,-0.45].
$$

### Bound Computation Workflow

- implement the one-step dynamics as a nonlinear function;
- create symbolic input and output variables and an `ABCrownSolver`;
- encode the input set with `IOConstraints`;
- call `compute_bounds(...)` and extract the returned `lower` and `upper`
  tensors.

In [ ]:
# Step 1: implement the one-step dynamics as a nonlinear function.
class VanDerPolStep(nn.Module):
    def forward(self, state):
        # state has shape (batch, 2), with columns q and v.
        q = state[:, 0:1]
        v = state[:, 1:2]

        # One explicit Euler step for the Van der Pol oscillator with dt=0.1.
        q_next = q + 0.1 * v
        v_next = v + 0.1 * ((1.0 - q.square()) * v - q)

        # The verifier sees the next state y=(q_next, v_next).
        return torch.cat((q_next, v_next), dim=1)

reach_model = VanDerPolStep().to(DEVICE).eval()

# Step 2: create symbolic input and output variables and the solver.
reach_x = input_vars(2)
reach_y = output_vars(2)

# The one-iteration setting returns the initial CROWN bound quickly.
reach_config = ConfigBuilder.from_defaults().set("bab/max_iterations", 1)
reach_solver = ABCrownSolver(reach_model, reach_x, reach_y, config=reach_config)

# Step 3: specify the input set with IOConstraints.
# IOConstraints accepts ordinary CPU tensors for input bounds.
reach_lower = torch.tensor([-3.0, -0.55])
reach_upper = torch.tensor([3.0, -0.45])
reach_domain = IOConstraints(
    input_vars=reach_x,
    input_constraint=(reach_x >= reach_lower) & (reach_x <= reach_upper),
)

### Bound and Plot the One-Step Image

`compute_bounds(...)` returns coordinatewise lower and upper values for
$f(\mathcal{X})$. Sampled successors lie inside the sound outer rectangle.

In [ ]:
# Step 4: call compute_bounds(...) on the requested output coordinates.
reach_result = reach_solver.compute_bounds(
    constraints=reach_domain,
    objective=reach_y,
)

# compute_bounds(...) returns lower and upper tensors for the requested outputs.
reachable_lower = reach_result.lower
reachable_upper = reach_result.upper

print("lower:", reachable_lower)
print("upper:", reachable_upper)

plot_reachability(
    reach_model,
    reach_lower,
    reach_upper,
    reachable_lower,
    reachable_upper,
)

## Application 2: Lyapunov Verification

Lyapunov verification applies $\alpha,\!\beta$-CROWN to nonlinear
satisfiability queries for Lyapunov conditions.

### Example 2A: Discrete-Time Pendulum

For the closed-loop pendulum

$$
x^+=g_d(x)=f_d(x,\pi(x)),
$$

define the sublevel set

$$
\Omega_\rho=\{x:V(x)\leq\rho\}.
$$

Over the verification domain $\mathcal{D}$, the certificate requires states
with $V(x)\leq\rho$ to satisfy Lyapunov decrease and forward invariance:

$$
V(x)\leq\rho
\Longrightarrow
\left[
(1-\kappa)V(x)-V(g_d(x))>-\varepsilon
\;\land\;
g_d(x)\in\mathcal{D}
\right].
$$

(Discrete-time formulation: Yang*, Dai*, Shi, Hsieh, Tedrake, and Zhang,
[*Lyapunov-stable Neural Control for State and Output Feedback: A Novel
Formulation*](https://arxiv.org/abs/2404.07956), ICML 2024.)

<p align="center">
  <img
    src="https://github.com/Verified-Intelligence/abCROWN_Control_Tutorial/blob/main/figures/tutorial_lyapunov_problem.png?raw=1"
    alt="Lyapunov stability and invariance problem"
    width="46%"
  />
</p>

### Step 1. Implement the Closed-Loop Certificate Function

The verified nonlinear function combines the discrete-time pendulum, neural controller,
quadratic Lyapunov function, and verifier-facing output vector.

#### Pendulum Dynamics

For $x=(\theta,\dot\theta)$ and torque $u=\pi(x)$,

$$
\ddot\theta=
-\frac{\beta}{ml^2}\dot\theta
+\frac{g}{l}\sin(\theta)
+\frac{u}{ml^2}.
$$

Forward Euler integration produces the discrete-time transition.

In [ ]:
class DiscretePendulumDynamics(nn.Module):
    def __init__(self, m=0.15, l=0.5, beta=0.1, g=9.81, dt=0.05):
        super().__init__()

        # Physical parameters and integration step.
        self.m = m
        self.l = l
        self.beta = beta
        self.g = g
        self.dt = dt

    def forward(self, x, u):
        # x = (theta, theta_dot), u = torque.
        theta = x[:, 0:1]
        theta_dot = x[:, 1:2]

        # Moment term m*l^2 appears in damping and torque scaling.
        ml2 = self.m * self.l * self.l

        # Continuous-time angular acceleration.
        theta_ddot = (
            (-self.beta / ml2) * theta_dot
            + (self.g / self.l) * torch.sin(theta)
            + u / ml2
        )

        # Forward Euler discretization x^+=f_d(x,u).
        theta_next = theta + self.dt * theta_dot
        theta_dot_next = theta_dot + self.dt * theta_ddot
        return torch.cat((theta_next, theta_dot_next), dim=1)

#### Neural Controller

The controller is a clamped multilayer perceptron with bounded torque output.

In [ ]:
class ClampMLPController(nn.Module):
    def __init__(self, in_dim=2, hidden_dim=8, u_lower=-0.75, u_upper=0.75):
        super().__init__()

        # Actuator saturation bounds for the torque command.
        self.u_lower = float(u_lower)
        self.u_upper = float(u_upper)

        # Equilibrium is registered as a buffer so it moves with .to(DEVICE).
        self.register_buffer("x_equilibrium", torch.zeros(in_dim))

        # Small MLP controller pi(x).
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        # Subtract pi(0) so the controller is centered at the equilibrium.
        raw = self.net(x) - self.net(self.x_equilibrium)

        # Enforce the torque constraint u in [u_lower, u_upper].
        return torch.clamp(raw, min=self.u_lower, max=self.u_upper)

#### Quadratic Lyapunov Function

The candidate

$$
V(x)=x^\top(\epsilon I+R^\top R)x
$$

satisfies $V(0)=0$ and $V(x)>0$ for every $x\neq0$.

In [ ]:
class QuadraticLyapunov(nn.Module):
    def __init__(self, x_dim=2, r_rows=2, epsilon=0.01):
        super().__init__()

        # epsilon*I makes the quadratic form positive definite.
        self.x_dim = x_dim
        self.epsilon = epsilon
        self.register_buffer("goal_state", torch.zeros(x_dim))

        # R is learned; epsilon*I + R^T R remains positive definite.
        self.R = nn.Parameter(torch.zeros(r_rows, x_dim))

    def forward(self, x):
        # Measure the state relative to the equilibrium.
        centered = x - self.goal_state

        # Construct Q = epsilon*I + R^T R on the same device as x.
        identity = torch.eye(self.x_dim, device=x.device)
        matrix = self.epsilon * identity + self.R.T @ self.R

        # Return V(x)=x^T Q x as a scalar output with shape (batch, 1).
        return torch.sum(centered * (centered @ matrix), dim=1, keepdim=True)

#### Verifier-Facing Condition

The closed-loop certificate is

$$
V(x)\leq\rho
\Longrightarrow
\left[
(1-\kappa)V(x)-V(g_d(x))>-\varepsilon
\;\land\;
g_d(x)\in\mathcal{D}
\right].
$$

The composed nonlinear function exposes the quantities needed to encode this implication:

$$
y_0=(1-\kappa)V(x)-V(x^+),\quad
y_1=V(x),\quad
(y_2,y_3)=x^+.
$$

These outputs make the decrease margin, sublevel-set condition, and
forward-invariance condition available as symbolic output constraints.

In [ ]:
class DiscreteLyapunovCondition(nn.Module):
    def __init__(self, dynamics, controller, lyapunov, kappa=0.001):
        super().__init__()

        # The verifier analyzes the composed closed-loop map.
        self.dynamics = dynamics
        self.controller = controller
        self.lyapunov = lyapunov
        self.kappa = kappa

    def forward(self, x):
        # Current Lyapunov value V(x).
        value = self.lyapunov(x)

        # Closed-loop control u=pi(x) and one-step successor x^+=f_d(x,u).
        control = self.controller(x)
        x_next = self.dynamics(x, control)

        # Next Lyapunov value V(x^+).
        next_value = self.lyapunov(x_next)

        # Positive decrease_margin proves V(x^+) <= (1-kappa)V(x).
        decrease_margin = (1.0 - self.kappa) * value - next_value

        # Output order is fixed for the symbolic property:
        # y0=decrease margin, y1=V(x), y2=theta^+, y3=theta_dot^+.
        return torch.cat((decrease_margin, value, x_next), dim=1)

#### Load Model Weights

The loaded weights define the neural controller and Lyapunov matrix used in the
closed-loop verification problem.

In [ ]:
dynamics = DiscretePendulumDynamics()
controller = ClampMLPController()
lyapunov = QuadraticLyapunov()

lyapunov_condition = DiscreteLyapunovCondition(
    dynamics,
    controller,
    lyapunov,
).to(DEVICE)

lyapunov_condition = load_pendulum_lyapunov_model(lyapunov_condition)

### Step 2. Create Symbolic Variables and Solver

`input_vars(2)` represents the state $x=(\theta,\dot\theta)$. `output_vars(4)`
represents the four verifier-facing outputs

$$
y_0=(1-\kappa)V(x)-V(x^+),\qquad
y_1=V(x),\qquad
y_2=\theta^+,\qquad
y_3=\dot\theta^+.
$$

In [ ]:
# Symbolic state x=(theta, theta_dot) and four verifier-facing outputs.
lyapunov_x = input_vars(2)
lyapunov_y = output_vars(4)

# Default configuration is sufficient for this trained certificate.
lyapunov_solver = ABCrownSolver(
    lyapunov_condition,
    lyapunov_x,
    lyapunov_y,
    config=ConfigBuilder.from_defaults(),
)

### Step 3. Specify Constraints

The domain is $\mathcal{D}=[-12,12]^2$. The implication is written as the
equivalent disjunction

$$
V(x)>\rho
\;\lor\;
\left[
y_0>-\varepsilon
\land x^+\in\mathcal{D}
\right].
$$

The input formula states $x\in\mathcal{D}$.

In [ ]:
# Property constants:
#   rho: Lyapunov sublevel threshold.
#   decrease_tolerance: small numerical slack in the strict decrease margin.
rho = 75.2487
decrease_tolerance = 1e-6

# Verification domain D=[-12,12]^2.
domain_limit = 12.0
domain_lower = torch.tensor([-domain_limit, -domain_limit], device=DEVICE)
domain_upper = torch.tensor([domain_limit, domain_limit], device=DEVICE)

# Encode:
#   input: x in D
#   output: V(x)>rho OR [decrease_margin>-eps AND x^+ in D]
lyapunov_property = IOConstraints(
    input_vars=lyapunov_x,
    output_vars=lyapunov_y,
    input_constraint=(
        (lyapunov_x >= domain_lower) & (lyapunov_x <= domain_upper)
    ),
    output_constraint=(lyapunov_y[1] > rho) | (
        (lyapunov_y[0] > -decrease_tolerance)
        & (lyapunov_y[2] > -domain_limit)
        & (lyapunov_y[2] < domain_limit)
        & (lyapunov_y[3] > -domain_limit)
        & (lyapunov_y[3] < domain_limit)
    ),
)

### Step 4. Run `verify(...)`

`verify(...)` returns a `SolveResult` with `status`:

- `verified`: the property was proved over the full domain;
- `falsified`: a state in $\mathcal{D}$ violating the implication was found;
- `unknown`: the configured resource budget was inconclusive.

For this Lyapunov condition, `verified` proves that every $x\in\mathcal{D}$
with $V(x)\leq\rho$ satisfies the decrease condition and $x^+\in\mathcal{D}$.

In [ ]:
# verify(...) attempts to prove the implication for every x in D.
lyapunov_result = lyapunov_solver.verify(constraints=lyapunov_property)

# A verified result proves decrease and x^+ in D whenever x in D and V(x) <= rho.
print("status:", lyapunov_result.status)

### Example 2B: Continuous-Time Van der Pol

For controlled continuous dynamics $\dot x=f(x,\pi(x))$, Lyapunov decrease
uses

$$
\dot V(x)=
\nabla V(x)^\top f(x,\pi(x)).
$$

The main verification challenge is the Jacobian term $\nabla V(x)$.
The verifier supports Jacobian terms by inserting the `JacobianOP` identifier
into the computation graph. Gradient-containing computation graphs are
automatically expanded for bound propagation.

(Continuous-time example: Li*, Zhong*, Hu, and Zhang,
[*Two-Stage Learning of Stabilizing Neural Controllers via Zubov Sampling and
Iterative Domain Expansion*](https://arxiv.org/abs/2506.01356), NeurIPS 2025.)

In [ ]:
# Step 1: implement V_dot with JacobianOP.
# JacobianOP is provided by auto_LiRPA, the underlying bound-propagation package.
from auto_LiRPA.jacobian import JacobianOP

class ContinuousTimeLyapunovCondition(nn.Module):
    def __init__(self, dynamics, controller, lyapunov):
        super().__init__()
        self.dynamics = dynamics
        self.controller = controller
        self.lyapunov = lyapunov

    def forward(self, x):
        # The input must require gradients for JacobianOP(value, x).
        x = x.clone().requires_grad_(True)

        value = self.lyapunov(x)
        control = self.controller(x)
        vector_field = self.dynamics.f_torch(x, control)

        # Shape: (batch, 1, state_dim) -> (batch, state_dim).
        gradient = JacobianOP.apply(value, x).squeeze(1)

        # V_dot = grad V(x)^T f(x, pi(x)).
        derivative = torch.sum(gradient * vector_field, dim=1, keepdim=True)

        return torch.cat((value, derivative), dim=1)

### Continuous-Time Verification

The property to prove is

$$
\forall x\in\mathcal{D},\quad
V_{\min}\leq V(x)\leq V_{\max}
\Longrightarrow
\dot V(x)<\varepsilon.
$$

Equivalently, for all $x\in\mathcal{D}$,

$$
V(x)<V_{\min}\;\lor\;V(x)>V_{\max}\;\lor\;\dot V(x)<\varepsilon.
$$

`IOConstraints` encodes $x\in\mathcal{D}$ as the input formula and the
remaining disjunction as the symbolic output formula.

In [ ]:
# Step 2: load the closed-loop certificate function and create the solver.
continuous_condition = build_continuous_vanderpol_lyapunov_model(
    ContinuousTimeLyapunovCondition,
)

continuous_x = input_vars(2)
continuous_y = output_vars(2)
continuous_solver = ABCrownSolver(
    continuous_condition,
    continuous_x,
    continuous_y,
)

# Step 3: specify the state domain and Lyapunov derivative property.
v_min = 0.0106
v_max = 0.989
v_dot_tolerance = 3e-6
continuous_property = IOConstraints(
    input_vars=continuous_x,
    output_vars=continuous_y,
    input_constraint=(continuous_x >= [-4.8, -10.8])
    & (continuous_x <= [4.8, 10.8]),
    output_constraint=(continuous_y[0] < v_min)
    | (continuous_y[0] > v_max)
    | (continuous_y[1] < v_dot_tolerance),
)

# Step 4: run verify(...).
continuous_result = continuous_solver.verify(constraints=continuous_property)
print("continuous status:", continuous_result.status)

## Application 3: Control Optimization
### Finite-Horizon Objective

A four-step acceleration sequence controls a damped nonlinear point mass:

$$
p_{t+1}=p_t+\Delta t\,v_t,
$$

$$
v_{t+1}=v_t+\Delta t\left(u_t-c_vv_t-k\sin(p_t)\right).
$$

Starting from fixed $(p_0,v_0)$ and target $(p^\star,v^\star)$, the objective
combines terminal position error, terminal velocity error, and control effort:

$$
J(u)=J_p+0.35J_v+0.08J_u,
$$

with

$$
J_p=(p_4-p^\star)^2,\qquad
J_v=(v_4-v^\star)^2,\qquad
J_u=\sum_{t=0}^{3}|u_t|.
$$

In [ ]:
class SimpleMPCDynamics(nn.Module):
    def __init__(self, horizon=4, dt=0.30, damping=0.24, strength=0.50):
        super().__init__()
        self.horizon = horizon
        self.dt = dt
        self.damping = damping
        self.strength = strength

        self.p0 = -1.2
        self.v0 = 0.9
        self.p_target = 1.0
        self.v_target = 0.0

    def forward(self, u):
        batch = u.shape[0]
        p = torch.full((batch, 1), self.p0, dtype=u.dtype, device=u.device)
        v = torch.full((batch, 1), self.v0, dtype=u.dtype, device=u.device)
        effort_cost = torch.zeros_like(p)

        for t in range(self.horizon):
            u_t = u[:, t : t + 1]
            p = p + self.dt * v
            v = v + self.dt * (
                u_t - self.damping * v - self.strength * torch.sin(p)
            )
            effort_cost = effort_cost + torch.abs(u_t)

        position_cost = (p - self.p_target).square()
        velocity_cost = (v - self.v_target).square()
        return torch.cat((position_cost, velocity_cost, effort_cost), dim=1)

mpc_model = SimpleMPCDynamics().to(DEVICE).eval()

### Constrained Global Optimization

The decision variable is $u=(u_0,u_1,u_2,u_3)$ with
$u_t\in[-1.2,1.2]$. The symbolic objective is formed directly from the three
cost outputs.

In [ ]:
# Symbolic control sequence u0,u1,u2,u3 and cost components.
control_x = input_vars(4)
cost_y = output_vars(3)

# The solver optimizes over the constrained input domain below.
optimization_solver = ABCrownSolver(
    mpc_model,
    control_x,
    cost_y,
    config=ConfigBuilder.from_defaults(),
)

# Constraint on every control component.
control_constraints = IOConstraints(
    input_vars=control_x,
    input_constraint=(control_x >= -1.2) & (control_x <= 1.2),
)

# Scalar objective J = J_p + 0.35 J_v + 0.08 J_u.
cost_objective = cost_y[0] + 0.35 * cost_y[1] + 0.08 * cost_y[2]

### Solve the Optimization Problem

`minimize(...)` returns a feasible control sequence and the best objective
value found by the branch-and-bound search. `maximize(...)` supports the same
symbolic objective interface for maximization problems.

In [ ]:
# Solve the constrained nonlinear control optimization problem.
optimization_result = optimization_solver.minimize(
    objective=cost_objective,
    constraints=control_constraints,
    return_bound_history=True,
)

# primal_value is the best objective found; x_best is the associated control.
optimization_primal_value = optimization_result.primal_value
optimal_control = optimization_result.x_best

print("primal value:", optimization_primal_value)
print("control sequence:", optimal_control)

### Primal and Dual Bound Convergence

`optimization_result.bound_history` contains:

- `primal_values`: feasible objective values found during branch-and-bound;
- `dual_bounds`: certified lower bounds on the global optimum.

For minimization, the primal curve is an upper bound on the optimum. The gap
between the two curves is the remaining optimality gap.

In [ ]:
# Extract primal and dual bounds at each iteration.
optimization_history = optimization_result.bound_history
optimization_primal_values = optimization_history["primal_values"]
optimization_dual_bounds = optimization_history["dual_bounds"]

plot_optimization_bounds(
    optimization_primal_values,
    optimization_dual_bounds,
)

## Additional Interface: Nonlinear Satisfiability
### A dReal-Compatible Formula

dReal is a mature SMT solver for nonlinear real arithmetic. The abcrown SMT
interface is dReal-compatible; except for the import line, the formula code is
identical.

<p align="center">
  <img
    src="https://github.com/Verified-Intelligence/abCROWN_Control_Tutorial/blob/main/figures/tutorial_sat_problem.png?raw=1"
    alt="Nonlinear satisfiability problem"
    width="46%"
  />
</p>

#### dReal Reference Form

<pre><code class="language-python"># This import line is the only code change.
from dreal import *

two_pi = 6.283185307179586
x = Variable(&quot;x&quot;)
y = Variable(&quot;y&quot;)
z = Variable(&quot;z&quot;)

f_sat = And(0 &lt;= x, x &lt;= two_pi,
            0 &lt;= y, y &lt;= two_pi,
            -2 &lt;= z, z &lt;= 2,
            sin(x) + cos(y) == z)

result = CheckSatisfiability(f_sat, 0.001)
print(result)</code></pre>

In [ ]:
# This import line is the only code change.
from abcrown.abcrown_smt import *

two_pi = 6.283185307179586
x = Variable("x")
y = Variable("y")
z = Variable("z")

f_sat = And(0 <= x, x <= two_pi,
            0 <= y, y <= two_pi,
            -2 <= z, z <= 2,
            sin(x) + cos(y) == z)

result = CheckSatisfiability(f_sat, 0.001)
print(result)